# Phase 4C.3 — Calibrated Late Fusion (development only) — Colab launcher

Runs the prospectively locked protocol `ml/configs/calibrated_late_fusion_protocol.yaml`:
visual (576-d) and DSP (16-d) logistic scorers on the **same 341 development sources and the same
5x4 source-grouped folds as the Visual/DSP ablation**, per-modality temperature scaling fitted on
inner out-of-fold scores only, and a two-input logistic stacker (`late_fusion_stacked`, primary).

* Default `MODE = "preflight"`: verifies inputs, fits nothing.
* `"pilot"`: outer fold 0 only (61 fits). `"full"`: 5 outer folds (305 fits); requires `ALLOW_FULL = True`.
* Reuses the ablation feature caches read-only when `ABLATION_OUTPUT` exists (preferred: paired predictions).
* Zero locked-test access. Results are development/exploratory and must not be reported as confirmatory.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

MODE = "preflight"   # "preflight" -> "pilot" -> "full"
ALLOW_FULL = False   # must be True for MODE == "full"
DEVICE = "cpu"       # only used if the visual cache has to be built ("cuda" or "cpu")

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/forensics-web-lab")
CODE_ARCHIVE = DRIVE_ROOT / "calibrated_late_fusion/inputs/phase_4c3_calibrated_late_fusion_code.tar.gz"
EXPECTED_SNAPSHOT_MANIFEST_SHA256 = ""  # snapshot_manifest_sha256 from research/evidence/phase-4c.3a/package_receipt.json
EXPECTED_CODE_SHA256 = ""  # optional archive_sha256 (gzip bytes can differ across zlib builds; the manifest hash cannot)
DATASET_ARCHIVE = DRIVE_ROOT / "phase_4c1/inputs/phase_4c1_binary_n250_reusable.tar"
ABLATION_OUTPUT = DRIVE_ROOT / "visual_dsp_ablation/runs/execution_ablation"
FALLBACK_WEIGHTS = DRIVE_ROOT / "phase_4c1/inputs/mobilenet_v3_small-047dcff4.pth"

STAGE = Path("/content/calibrated_late_fusion_stage")
BUNDLE = Path("/content/development_bundle")
OUTPUT = DRIVE_ROOT / "calibrated_late_fusion/runs/execution_late_fusion"
REPORT_DIR = DRIVE_ROOT / "calibrated_late_fusion/report"


In [ ]:
import hashlib, importlib.metadata, json, shutil, tarfile
from packaging.version import Version
from pathlib import PurePosixPath

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract(archive_path, destination, mode):
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True)
    with tarfile.open(archive_path, mode) as archive:
        members = archive.getmembers()
        for member in members:
            posix = PurePosixPath(member.name)
            assert member.isfile() or member.isdir(), f"Non-regular archive member: {member.name}"
            assert not posix.is_absolute() and ".." not in posix.parts and "\\" not in member.name
        archive.extractall(destination, members=members)

assert MODE in {"preflight", "pilot", "full"}
assert MODE != "full" or ALLOW_FULL, "MODE='full' requires ALLOW_FULL=True"
assert CODE_ARCHIVE.is_file(), f"Missing code archive: {CODE_ARCHIVE}"
assert DATASET_ARCHIVE.is_file(), f"Missing dataset archive: {DATASET_ARCHIVE}"

code_sha = sha256_file(CODE_ARCHIVE)
print("code archive sha256:", code_sha)
if EXPECTED_CODE_SHA256:
    assert code_sha == EXPECTED_CODE_SHA256, "Code archive does not match the expected SHA-256"

minimums = {"torch": "2.0.0", "numpy": "1.24.0", "scipy": "1.10.0", "Pillow": "9.5.0",
            "scikit-learn": "1.2.0", "PyYAML": "6.0"}
for package, minimum in minimums.items():
    actual = importlib.metadata.version(package)
    assert Version(actual) >= Version(minimum), f"{package} {actual} < {minimum}"

safe_extract(CODE_ARCHIVE, STAGE, "r:gz")
snapshot_sha = sha256_file(STAGE / "SNAPSHOT_MANIFEST.json")
print("snapshot manifest sha256:", snapshot_sha)
if EXPECTED_SNAPSHOT_MANIFEST_SHA256:
    assert snapshot_sha == EXPECTED_SNAPSHOT_MANIFEST_SHA256, "Snapshot manifest does not match the receipt"
snapshot = json.loads((STAGE / "SNAPSHOT_MANIFEST.json").read_text())
for member in snapshot["members"]:
    assert sha256_file(STAGE / member["path"]) == member["sha256"], member["path"]
print("source commit:", snapshot["source_commit"], "members:", len(snapshot["members"]))

safe_extract(DATASET_ARCHIVE, BUNDLE, "r:")
assert (BUNDLE / "manifest_pilot_a_option_p.csv").is_file(), "Manifest missing in extracted bundle"

weights_path = STAGE / "models/research/pretrained/mobilenet_v3_small-047dcff4.pth"
if not weights_path.is_file():
    weights_path = FALLBACK_WEIGHTS
REUSE_CACHES = (ABLATION_OUTPUT / "shared/visual_features.pt").is_file()
print("reuse ablation feature caches:", REUSE_CACHES, "| weights present:", weights_path.is_file())
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, "-m", "ml.training.run_calibrated_late_fusion",
    "--mode", MODE,
    "--protocol", str(STAGE / "ml/configs/calibrated_late_fusion_protocol.yaml"),
    "--manifest", str(BUNDLE / "manifest_pilot_a_option_p.csv"),
    "--data-root", str(BUNDLE),
    "--weights-path", str(weights_path),
    "--output-dir", str(OUTPUT),
    "--device", DEVICE,
]
if REUSE_CACHES:
    cmd += ["--feature-cache-dir", str(ABLATION_OUTPUT)]
print("Executing:", " ".join(cmd))
subprocess.run(cmd, cwd=STAGE, check=True)


In [ ]:
if MODE == "full":
    cmd = [
        sys.executable, str(STAGE / "scripts/research/analyze_calibrated_late_fusion.py"),
        "--output-dir", str(OUTPUT),
        "--report-dir", str(REPORT_DIR),
        "--protocol", str(STAGE / "ml/configs/calibrated_late_fusion_protocol.yaml"),
    ]
    if (ABLATION_OUTPUT / "fits/visual_dsp_fusion/outer_0/predictions.csv").is_file():
        cmd += ["--ablation-output-dir", str(ABLATION_OUTPUT)]
    print("Executing:", " ".join(cmd))
    subprocess.run(cmd, cwd=STAGE, check=True)
    print((REPORT_DIR / "ANALYSIS_REPORT.md").read_text())
else:
    for path in sorted((OUTPUT / "fits").glob("outer_*/fold_receipt.json")):
        receipt = json.loads(path.read_text())
        print(path.parent.name, {
            "C": {m: receipt["modalities"][m]["best_C"] for m in ("visual", "dsp")},
            "T": {m: round(receipt["modalities"][m]["temperature"]["temperature"], 4) for m in ("visual", "dsp")},
            "fits": receipt["fit_counts"]["total_fits"],
            "unconverged": receipt["fit_counts"]["unconverged_fits"],
            "seconds": round(receipt["fit_wall_seconds"], 2),
        })


In [ ]:
# Bundle the run receipts/predictions and the report for transfer back to the repository.
if MODE == "full":
    bundle = DRIVE_ROOT / "calibrated_late_fusion/phase_4c3_results.tar.gz"
    with tarfile.open(bundle, "w:gz") as archive:
        for path in sorted(OUTPUT.rglob("*")):
            if path.is_file() and "shared" not in path.relative_to(OUTPUT).parts:
                archive.add(path, arcname=f"run/{path.relative_to(OUTPUT).as_posix()}")
        for path in sorted(REPORT_DIR.rglob("*")):
            if path.is_file():
                archive.add(path, arcname=f"report/{path.relative_to(REPORT_DIR).as_posix()}")
    print("results bundle:", bundle, sha256_file(bundle))
